# Exploring Telco Customer Churn Dataset

This notebook will focus on exploring the telco customer churn dataset, analyzing the results of the exploration, and saving the processed data to train/test splits for later modeling.

### Load Data

In [45]:
import pandas as pd

churn = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

### Exploratory Analysis

The goal of the exploratory analysis is to get an idea of the contents of the data. A High-level overview of observations include:
* dataset contains 7043 customers
* mix of feature types, including numeric, binary, and multi-class
* minimal missing data
* significant class imbalance in target feature-we will need to carefully observe this throughout

In [46]:
churn.shape

(7043, 21)

We see that there are 7043 rows in our dataset with 21 features. One of those is our target feature, churn, so we have 20 potential features for training (for now). 

In [47]:
churn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [48]:
churn.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [50]:
churn['Churn'].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

This is a key insight–we have now determined a class imbalance between the binary churn classification. We will need to consider this as we split the data into train and test, as well as during any model evaluation and model monitoring. 

### Feature Engineering
We will look at all of the values of our features and perform feature engineering. Key feature issues we will explore include:
* missing data: deal with any null or missing values in the dataset
* feature type: identify discrepencies, and encode categorical variables

Based on these, the following actions are performed on the data:
* drop customerID: identification feature not needed for training
* convert TotalCharges from str to numeric
* converted multi-class features that are redundant and should be binary to binary and encode
* encoded multi-class and binary features for training

In [51]:
churn.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [52]:
(churn == " ").sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

We see no null values in any of our features, but the "TotalCharges" column has blank values. We will address these by converting the blanks to nulls, and filtering our data for null values.

In [53]:
churn["TotalCharges"] = pd.to_numeric(
    churn["TotalCharges"],
    errors="coerce"
)

In [54]:
churn["TotalCharges"].isnull().sum()

np.int64(11)

In [55]:
churn = churn.dropna()

Next, we will drop the customerID field as this is an identity-based field rather than a potential trainable feature. Categorical feature encoding will also be included, which does include coverting our target feature to numerical. We will need to identify which columns are non-numeric, determine whether they are binary or multi-categorical, and encode our features.

In [56]:
churn = churn.drop(columns=["customerID"])

In [57]:
churn["Churn"] = (
    churn["Churn"]
    .map({"Yes": 1, "No": 0})
)

In [58]:
churn["Churn"].value_counts(normalize=True)

Churn
0    0.734215
1    0.265785
Name: proportion, dtype: float64

In [59]:
churn.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [60]:
non_numeric_cols = ["gender",
                    "Partner",
                    "Dependents",
                    "PhoneService",
                    "MultipleLines",
                    "InternetService",
                    "Contract",
                    "PaymentMethod",
                    "OnlineSecurity",
                    "OnlineBackup",
                    "DeviceProtection",
                    "TechSupport",
                    "StreamingTV",
                    "StreamingMovies",
                    "PaperlessBilling",
                    ]

for c in non_numeric_cols:
    categories = list(churn[f'{c}'].unique())
    print(f"Num unique categories in {c}: {len(categories)} including {categories}")

Num unique categories in gender: 2 including ['Female', 'Male']
Num unique categories in Partner: 2 including ['Yes', 'No']
Num unique categories in Dependents: 2 including ['No', 'Yes']
Num unique categories in PhoneService: 2 including ['No', 'Yes']
Num unique categories in MultipleLines: 3 including ['No phone service', 'No', 'Yes']
Num unique categories in InternetService: 3 including ['DSL', 'Fiber optic', 'No']
Num unique categories in Contract: 3 including ['Month-to-month', 'One year', 'Two year']
Num unique categories in PaymentMethod: 4 including ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']
Num unique categories in OnlineSecurity: 3 including ['No', 'Yes', 'No internet service']
Num unique categories in OnlineBackup: 3 including ['Yes', 'No', 'No internet service']
Num unique categories in DeviceProtection: 3 including ['No', 'Yes', 'No internet service']
Num unique categories in TechSupport: 3 including ['No', 'Yes', 'No intern

Since internet service is a feature, we will convert the categories that contain the option "No internet service" to be binary features containing only Yes and No.

In [61]:
convert_to_binary = ['MultipleLines', 
                     'OnlineSecurity', 
                     'OnlineBackup', 
                     'DeviceProtection', 
                     'TechSupport', 
                     'StreamingTV', 
                     'StreamingMovies']

for col in convert_to_binary:
    churn[col] = churn[col].replace(
        {
            "Yes": 1,
            "No": 0,
            "No internet service": 0,
            "No phone service": 0
        }
    )

/var/folders/fh/3p72kl9960qd02pfdjfnkbh40000gn/T/ipykernel_70789/3458422613.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  churn[col] = churn[col].replace(


In [62]:
binary_cols = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

churn[binary_cols] = churn[binary_cols].replace(
    {"Yes": 1, "No": 0}
)

/var/folders/fh/3p72kl9960qd02pfdjfnkbh40000gn/T/ipykernel_70789/1283197277.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  churn[binary_cols] = churn[binary_cols].replace(


In [63]:
categorical_cols = [
    "gender",
    "InternetService",
    "Contract",
    "PaymentMethod",
]

churn = pd.get_dummies(
    churn,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [64]:
churn.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,...,TotalCharges,Churn,gender_Male,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,0,1,0,0,0,1,0,0,...,29.85,0,0,0,0,0,0,0,1,0
1,0,0,0,34,1,0,1,0,1,0,...,1889.50,0,1,0,0,1,0,0,0,1
2,0,0,0,2,1,0,1,1,0,0,...,108.15,1,1,0,0,0,0,0,0,1
3,0,0,0,45,0,0,1,0,1,1,...,1840.75,0,1,0,0,1,0,0,0,0
4,0,0,0,2,1,0,0,0,0,0,...,151.65,1,0,1,0,0,0,0,1,0


In [65]:
churn.describe()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,...,TotalCharges,Churn,gender_Male,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
count,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,...,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000
mean,0.162400,0.482509,0.298493,32.421786,0.903299,0.421928,0.286547,0.344852,0.343857,0.290102,...,2283.300441,0.265785,0.504693,0.440273,0.216155,0.209329,0.239619,0.216297,0.336320,0.228100
std,0.368844,0.499729,0.457629,24.545260,0.295571,0.493902,0.452180,0.475354,0.475028,0.453842,...,2266.771362,0.441782,0.500014,0.496455,0.411650,0.406858,0.426881,0.411748,0.472483,0.419637
min,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,18.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,9.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,401.450000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,29.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1397.475000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,1.000000,1.000000,55.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,3794.737500,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,72.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,8684.800000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Final Dataset Check

To verify data for readiness, we:
* verify to make sure there are no columns that have not been properly encoded
* chekc for missing values
* verify target encoding
* check final shape of data

In [66]:
churn.select_dtypes(include=["object"]).columns

Index([], dtype='object')

In [69]:
churn.isna().sum()

SeniorCitizen                            0
Partner                                  0
Dependents                               0
tenure                                   0
PhoneService                             0
MultipleLines                            0
OnlineSecurity                           0
OnlineBackup                             0
DeviceProtection                         0
TechSupport                              0
StreamingTV                              0
StreamingMovies                          0
PaperlessBilling                         0
MonthlyCharges                           0
TotalCharges                             0
Churn                                    0
gender_Male                              0
InternetService_Fiber optic              0
InternetService_No                       0
Contract_One year                        0
Contract_Two year                        0
PaymentMethod_Credit card (automatic)    0
PaymentMethod_Electronic check           0
PaymentMeth

In [70]:
churn['Churn'].value_counts(normalize=True)

Churn
0    0.734215
1    0.265785
Name: proportion, dtype: float64

In [71]:
churn.shape

(7032, 24)

Perfect! Data looks as expected. Now we can split data into train/test splits.

### Split into Train/Test Data
This will be pretty simple-split the data into two sections for training and testing. However, after discovering the disparity in the churn variable, we need to make sure to maintain this distribution by using a stratified split.

In [ ]:
# reordering columns-not explicitly necessary, but convenient for data cleanliness
target = "Churn"
features = [c for c in churn.columns if c != target]

df = churn[features + [target]]

In [ ]:
# saving full processed dataset
churn.to_csv("data/processed/final_dataset.csv", index=False)

In [67]:
from sklearn.model_selection import train_test_split

train_churn, test_churn = train_test_split(
    churn,
    test_size=0.2,
    stratify=churn["Churn"],
    random_state=42
)

In [68]:
train_churn.to_csv(
    "../data/processed/train.csv",
    index=False
)

test_churn.to_csv(
    "../data/processed/test.csv",
    index=False
)

Now we have cleaned our data and are ready for the next step-training our models.